# `manage_db` — TxGNN LaminDB instance setup

This notebook walks through every step required to initialise a fresh LaminDB
instance for the expanded TxGNN knowledge graph.

## Steps covered

| # | Task | Function |
|---|------|----------|
| 1 | Connect to the instance | `lamindb.connect()` |
| 2 | Pin bionty ontology source versions | `register_ontology_sources()` |
| 3 | Import bionty records into the instance | `bt.<Registry>.import_source()` |
| 4 | Register pertdb sources (laminlabs/pertdata) | `register_pertdb_sources()` |
| 5 | Query and transfer compounds from laminlabs/pertdata | `pt.Compound.connect()` |
| 6 | Inspect the custom `lnschema_txgnn` record types | — |
| 7 | Sync TxGNN nodes → LaminDB entities | `sync_txgnn_nodes_to_lamin_entities()` |

### Prerequisites

```bash
# Create and connect a LaminDB instance (run once)
lamin init --storage ./data/lamin_txgnn --modules bionty,pertdb,lnschema_txgnn
lamin connect myorg/myinstance
```

All functions live in `manage_db/` — import them directly.

In [1]:
import sys
from pathlib import Path

# Ensure the project root is on the path so manage_db is importable.
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import lamindb as ln
import bionty as bt
import pertdb as pt

# manage_db helpers
from manage_db.register_ontology_sources import (
    register_ontology_sources,
    register_pertdb_sources,
    print_results,
)
from manage_db.sync_nodes_to_lamindb import sync_txgnn_nodes_to_lamin_entities

→ connected lamindb: jkobject/mjouvencekb


## 1. Connect to the instance

Pass the instance slug (`"account/name"`). Use `lamin connect` on the CLI
to set a default so you can omit the slug here.

In [ ]:
INSTANCE = "jkobject/jouvencekb"   # ← change to your instance
#ln.connect(INSTANCE)

In [2]:
# Quick audit: how many records exist before we do anything?
REGISTRIES = [
    ("bt.Gene",                      bt.Gene),
    ("bt.Disease",                   bt.Disease),
    ("bt.Tissue",                    bt.Tissue),
    ("bt.Phenotype",                 bt.Phenotype),
    ("bt.Pathway",                   bt.Pathway),
    ("bt.CellType",                  bt.CellType),
    ("bt.CellLine",                  bt.CellLine),
    ("bt.Organism",                  bt.Organism),
    ("pt.Compound",                  pt.Compound),
    ("pt.Biologic",                  pt.Biologic),
    ("pt.GeneticPerturbation",       pt.GeneticPerturbation),
    ("pt.EnvironmentalPerturbation", pt.EnvironmentalPerturbation),
]

pd.DataFrame(
    [(name, reg.objects.count()) for name, reg in REGISTRIES],
    columns=["registry", "records"],
)

,registry,records
0,bt.Gene,0
1,bt.Disease,0
2,bt.Tissue,0
3,bt.Phenotype,0
4,bt.Pathway,0
5,bt.CellType,0
6,bt.CellLine,0
7,bt.Organism,0
8,pt.Compound,0
9,pt.Biologic,0


## 6. Custom `lnschema_txgnn` record types

Five node types in the TxGNN KG are not covered by bionty or pertdb.  They
are registered as custom `SQLRecord` subclasses in `manage_db/lnschema_txgnn/`:

| Model | Node type | Primary ID |
|---|---|---|
| `Paper` | `paper` | PubMed ID / DOI |
| `Transcript` | `transcript` | Ensembl ENST… |
| `Enhancer` | `enhancer` | ENCODE ID |
| `Dataset` | `dataset` | Internal UUID / DOI |
| `Mutation` | `mutation` | dbSNP rsID / HGVS |

> CellLine is already covered by `bionty.CellLine` (Cellosaurus).

In [3]:
# The module must be installed as a schema module in the LaminDB instance.
# If lamin init was called with --modules lnschema_txgnn, the tables already exist.

# Import and inspect the models.
import sys
sys.path.insert(0, str(ROOT / "manage_db"))
import lnschema_txgnn as txgnn_schema

for model in [txgnn_schema.Paper, txgnn_schema.Transcript,
              txgnn_schema.Enhancer, txgnn_schema.Dataset, txgnn_schema.Mutation]:
    fields = [f.name for f in model._meta.get_fields() if hasattr(f, 'column')]
    print(f"  {model.__name__:<15}  fields: {', '.join(fields)}")

  Paper            fields: id, branch, created_on, space, is_locked, _aux, created_at, created_by, run, updated_at, uid, pmid, doi, pmc_id, arxiv_id, title, year, journal, abstract
  Transcript       fields: id, branch, created_on, space, is_locked, _aux, created_at, created_by, run, updated_at, uid, ensembl_transcript_id, refseq_mrna, ccds_id, ensembl_gene_id, biotype, is_canonical
  Enhancer         fields: id, branch, created_on, space, is_locked, _aux, created_at, created_by, run, updated_at, uid, encode_id, ensembl_regulatory_id, encode_experiment_id, chromosome, start_pos, end_pos
  Dataset          fields: id, branch, created_on, space, is_locked, _aux, created_at, created_by, run, updated_at, uid, name, doi, description, version, source_url
  Mutation         fields: id, branch, created_on, space, is_locked, _aux, created_at, created_by, run, updated_at, uid, rsid, hgvs, clinvar_id, gnomad_id, chromosome, position, ref_allele, alt_allele, consequence


In [4]:
# Example: create a Paper record.
paper = txgnn_schema.Paper(
    pmid="37778123",
    doi="10.1038/s41591-023-02558-7",
    title="Prediction of drug-disease associations using graph neural networks",
    year=2023,
    journal="Nature Medicine",
).save()

print("Saved Paper uid:", paper.uid)
print(txgnn_schema.Paper.filter(pmid="37778123").one())

Saved Paper uid: ldwctaTpf66f
Paper(uid='ldwctaTpf66f', pmid='37778123', doi='10.1038/s41591-023-02558-7', pmc_id=None, arxiv_id=None, title='Prediction of drug-disease associations using graph neural networks', year=2023, journal='Nature Medicine', abstract=None, branch_id=1, created_on_id=1, space_id=1, created_by_id=1, run_id=None, created_at=2026-03-06 15:29:08 UTC, is_locked=False)


In [5]:
# Record counts in each custom registry.
pd.DataFrame([
    {"model": m.__name__, "records": m.objects.count()}
    for m in [txgnn_schema.Paper, txgnn_schema.Transcript,
               txgnn_schema.Enhancer, txgnn_schema.Dataset, txgnn_schema.Mutation]
])

,model,records
0,Paper,1
1,Transcript,0
2,Enhancer,0
3,Dataset,0
4,Mutation,0


## 7. Sync TxGNN nodes → LaminDB entities

`sync_txgnn_nodes_to_lamin_entities()` reads the legacy `nodes.tab` file,
maps each node to its canonical bionty or pertdb registry, looks up
existing records, and creates missing ones.

### Input format (`nodes.tab`)

| Column | Example |
|---|---|
| `node_index` | 0 |
| `node_id` | 51177 |
| `node_type` | gene/protein |
| `node_name` | ANAMORSIN |
| `node_source` | NCBI |

### Node type mapping

| Legacy type | Registry |
|---|---|
| `gene/protein` | `bionty.Gene` (NCBI gene ID) |
| `drug` | `pertdb.Compound` (DrugBank ID) |
| `disease` | `bionty.Disease` (MONDO ID) |
| `effect/phenotype` | `bionty.Phenotype` (HP ID) |
| `anatomy` | `bionty.Tissue` (UBERON ID) |
| `pathway` | `bionty.Pathway` (GO / Reactome ID) |
| `exposure` | `pertdb.EnvironmentalPerturbation` |
| `biological_process` | `bionty.Pathway` (GO ID) |
| `molecular_function` | `bionty.Pathway` (GO ID) |
| `cellular_component` | `bionty.Pathway` (GO ID) |

In [6]:
# Dry-run — inspect the mapping without touching the database.
mapping_dry = sync_txgnn_nodes_to_lamin_entities(
    nodes_path=ROOT / "data" / "txdata" / "nodes.tab",
    mapping_output_path=None,   # skip writing a CSV
    dry_run=True,
)

print("Shape:", mapping_dry.shape)
print()
print("Status breakdown:")
print(mapping_dry["status"].value_counts().to_string())
print()
print("Registry breakdown:")
print(mapping_dry["registry"].value_counts().to_string())

... synchronizing df_human__ensembl__release-114__Gene.parquet: 100.0%
Shape: (129375, 10)

Status breakdown:
status
missing      128108
uncertain      1267

Registry breakdown:
registry
bionty.Pathway                      46503
bionty.Gene                         27671
bionty.Disease                      17080
bionty.Phenotype                    15311
bionty.Tissue                       14035
pertdb.Compound                      7957
pertdb.EnvironmentalPerturbation      818


In [ ]:
# Peek at uncertain nodes (no ontology ID → cannot reliably match).
uncertain = mapping_dry[mapping_dry["status"] == "uncertain"]
print(f"Uncertain nodes: {len(uncertain):,}")
uncertain[["node_type", "node_name", "node_source", "registry", "key_field"]].head(10)

In [ ]:
# Run the real sync (writes records to the database).
# This may take 5–15 minutes depending on instance size.
mapping = sync_txgnn_nodes_to_lamin_entities(
    nodes_path=ROOT / "data" / "txdata" / "nodes.tab",
    mapping_output_path=ROOT / "data" / "txdata" / "node_entity_mapping.csv",
    dry_run=False,
)

print("Status breakdown:")
print(mapping["status"].value_counts().to_string())

In [ ]:
# Per-node-type coverage table.
summary = (
    mapping
    .groupby(["node_type", "status"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda df: df.sum(axis=1))
    .sort_values("total", ascending=False)
)
summary

In [ ]:
# Spot-check: verify a mapped disease node resolves in the LaminDB registry.
sample_disease = mapping[
    (mapping["node_type"] == "disease") & (mapping["status"] == "existing")
].iloc[0]

print("Node      :", sample_disease[["node_name", "node_id", "registry", "key_value"]])

disease_record = bt.Disease.filter(
    ontology_id=sample_disease["key_value"]
).one_or_none()
print("LaminDB   :", disease_record)

## Final record counts

After all steps above, the instance should contain:

In [ ]:
ln.connect(INSTANCE)

final = [
    ("bt.Gene",                      bt.Gene),
    ("bt.Disease",                   bt.Disease),
    ("bt.Tissue",                    bt.Tissue),
    ("bt.Phenotype",                 bt.Phenotype),
    ("bt.Pathway",                   bt.Pathway),
    ("bt.CellType",                  bt.CellType),
    ("bt.CellLine",                  bt.CellLine),
    ("bt.Organism",                  bt.Organism),
    ("pt.Compound",                  pt.Compound),
    ("pt.Biologic",                  pt.Biologic),
    ("pt.GeneticPerturbation",       pt.GeneticPerturbation),
    ("pt.EnvironmentalPerturbation", pt.EnvironmentalPerturbation),
]

pd.DataFrame(
    [(name, reg.objects.count()) for name, reg in final],
    columns=["registry", "records"],
)